In [6]:
# =========================
# INSTALAÇÃO DE BIBLIOTECAS
# =========================
# O 'pip install' baixa e instala as ferramentas externas necessárias:
# - pandas: para organizar os dados em formato de tabela.
# - openpyxl: para criar, editar e colocar gráficos no arquivo Excel.
# - yfinance: para conectar com o Yahoo Finance e baixar os dados da bolsa.
!pip install pandas openpyxl yfinance

import pandas as pd
import yfinance as yf
from openpyxl import load_workbook
from openpyxl.chart import LineChart, Reference

# =========================
# CONFIGURAÇÃO
# =========================
# Define qual ação será analisada (Petrobras) e o período de tempo (6 meses)
ativo = "PETR4.SA"
periodo = "6mo"

# =========================
# BAIXAR DADOS
# =========================
# Usa a biblioteca yfinance para buscar o histórico de preços do ativo na bolsa
df = yf.download(ativo, period=periodo)

# Verifica se os dados foram baixados. Se a tabela estiver vazia, interrompe o código com erro.
if df.empty:
  raise Exception("Erro ao baixar dados")

# 🔥 CORREÇÃO PRINCIPAL: remover MultiIndex
# O Yahoo Finance às vezes traz as colunas em formato duplo. Isso as transforma em colunas simples para evitar erros.
if isinstance(df.columns, pd.MultiIndex):
  df.columns = df.columns.get_level_values(0)

# Transforma a data, que era apenas um índice, em uma coluna normal da tabela
df.reset_index(inplace=True)

# =========================
# SIMULAÇÃO
# =========================
# Configura as variáveis iniciais para a simulação de investimento
aporte_mensal = 500
cotas = 0
carteira = []

# Cria um laço de repetição para passar por todos os dias da tabela
for i in range(len(df)):
  # Pega o preço de fechamento da ação no dia atual
  preco = float(df["Close"].iloc[i])

  # A cada 21 dias úteis (aproximadamente 1 mês), faz um novo aporte de R$ 500
  if i % 21 == 0:
    cotas += aporte_mensal / preco # Calcula quantas ações comprou com o dinheiro

  # Adiciona à lista o valor total que a pessoa tem investido naquele dia
  carteira.append(cotas * preco)

# Adiciona a lista gerada como uma nova coluna na tabela do Pandas
df["Carteira"] = carteira

# =========================
# SALVAR EXCEL
# =========================
# Exporta a tabela formatada para um arquivo Excel
arquivo = "relatorio_financeiro.xlsx"
df.to_excel(arquivo, index=False)

# =========================
# CRIAR GRÁFICOS
# =========================
# Abre o arquivo Excel gerado para podermos editá-lo
wb = load_workbook(arquivo)
ws = wb.active # Seleciona a aba principal do Excel
max_linha = len(df) + 1 # Identifica quantas linhas de dados existem para o gráfico saber onde parar

# Cria um dicionário para achar automaticamente o número de cada coluna pelo nome dela
colunas = {cell.value: idx+1 for idx, cell in enumerate(ws[1])}

col_data = colunas["Date"]
col_close = colunas["Close"]
col_carteira = colunas["Carteira"]

# --- Gráfico de Preço do Ativo ---
graf1 = LineChart()
graf1.title = "Preço do Ativo"
# Seleciona as células do Excel que contém os preços e as datas
dados = Reference(ws, min_col=col_close, min_row=1, max_row=max_linha)
datas = Reference(ws, min_col=col_data, min_row=2, max_row=max_linha)
graf1.add_data(dados, titles_from_data=True)
graf1.set_categories(datas)
# Insere o gráfico na célula H2
ws.add_chart(graf1, "H2")

# --- Gráfico da Evolução da Carteira ---
graf2 = LineChart()
graf2.title = "Evolução da Carteira"
# Seleciona as células do Excel que contém o valor total da carteira simulada
dados2 = Reference(ws, min_col=col_carteira, min_row=1, max_row=max_linha)
graf2.add_data(dados2, titles_from_data=True)
graf2.set_categories(datas)
# Insere o gráfico na célula H20
ws.add_chart(graf2, "H20")

# Salva as modificações finais no Excel
wb.save(arquivo)
print("✅ Excel com gráficos criado!")

# =========================
# DOWNLOAD
# =========================
# Comandos do Google Colab que forçam o arquivo a ser baixado para o seu computador
from google.colab import files
files.download(arquivo)

/tmp/ipykernel_18875/2980998824.py:27: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ativo, period=periodo)
[*********************100%***********************]  1 of 1 completed


✅ Excel com gráficos criado!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>